In [3]:
import yfinance as yf
import time
import warnings
warnings.filterwarnings('ignore')

In [6]:
START = '1995-01-01'
results = {}

def fetch_yf(name, ticker, delay=2):
    time.sleep(delay)
    try:
        d = yf.download(ticker, start=START, progress=False, auto_adjust=True)
        d = d.dropna(how='all')
        if len(d) > 0:
            results[name] = {
                'ticker': ticker,
                'start':  str(d.index[0].date()),
                'end':    str(d.index[-1].date()),
                'rows':   len(d),
                'status': 'OK'
            }
            print(f"{name:10s} {ticker:15s}: {d.index[0].date()} -> {d.index[-1].date()} ({len(d):,} rows)")
        else:
            results[name] = {'ticker': ticker, 'status': 'NO DATA'}
            print(f"{name:10s} {ticker:15s}: NO DATA")
    except Exception as e:
        results[name] = {'ticker': ticker, 'status': 'ERROR'}
        print(f"{name:10s} {ticker:15s}: ERROR")

# equity 
print('\n=== EQUITY INDICES ===')
fetch_yf('SPX',   '^GSPC')
fetch_yf('STOXX', '^STOXX50E')
fetch_yf('FTSE',  '^FTSE')
fetch_yf('N225',  '^N225')

# Vol
print('\n=== VOL INDICES ===')
fetch_yf('VIX',   '^VIX')
fetch_yf('MOVE',  '^MOVE')
fetch_yf('EUVIX', '^EUVIX')

# commodity
print('\n=== COMMODITIES ===')
fetch_yf('GOLD',  'GC=F')


=== EQUITY INDICES ===
SPX        ^GSPC          : 1995-01-03 -> 2026-05-15 (7,895 rows)
STOXX      ^STOXX50E      : 2007-03-30 -> 2026-05-15 (4,792 rows)
FTSE       ^FTSE          : 1995-01-03 -> 2026-05-15 (7,923 rows)
N225       ^N225          : 1995-01-04 -> 2026-05-14 (7,689 rows)

=== VOL INDICES ===
VIX        ^VIX           : 1995-01-03 -> 2026-05-15 (7,895 rows)
MOVE       ^MOVE          : 2002-11-12 -> 2026-05-14 (5,811 rows)
EUVIX      ^EUVIX         : 2020-05-15 -> 2020-05-15 (1 rows)

=== COMMODITIES ===
GOLD       GC=F           : 2000-08-30 -> 2026-05-15 (6,451 rows)


In [9]:
from dotenv import load_dotenv
from pathlib import Path
import os
import requests
import time

load_dotenv(dotenv_path=Path('../.env'))
FRED_API_KEY = os.getenv('FRED_API_KEY')
results = {}

def fetch_fred(name, series, delay=1):
    time.sleep(delay)
    try:
        url = (
            f'https://api.stlouisfed.org/fred/series/observations'
            f'?series_id={series}'
            f'&api_key={FRED_API_KEY}'
            f'&file_type=json'
            f'&observation_start=1990-01-01'
            f'&limit=100000'
            f'&sort_order=asc'
        )
        r = requests.get(url, timeout=15)
        data = r.json()
        if 'error_code' in data:
            print(f"{name:20s} {series:25s}: API ERROR — {data.get('error_message')}")
            return
        obs = [o for o in data['observations'] if o['value'] != '.']
        if obs:
            first = obs[0]['date']
            last  = obs[-1]['date']
            results[f'FRED_{name}'] = {
                'series': series,
                'start':  first,
                'end':    last,
                'rows':   len(obs),
                'status': 'OK'
            }
            print(f"{name:20s} {series:25s}: {first} -> {last} ({len(obs):,} rows)")
        else:
            print(f"{name:20s} {series:25s}: NO DATA")
    except Exception as e:
        print(f"{name:20s} {series:25s}: ERROR — {str(e)[:40]}")

print('=== TARGET PAIRS ===')
fetch_fred('EURUSD',    'DEXUSEU')
fetch_fred('GBPUSD',    'DEXUSUK')
fetch_fred('JPYUSD',    'DEXJPUS')

print('\n=== POLICY RATES ===')
fetch_fred('FED_FUNDS', 'FEDFUNDS')

print('\n=== US YIELDS ===')
fetch_fred('US_10Y',    'DGS10')
fetch_fred('US_2Y',     'DGS2')
fetch_fred('DTB3',      'DTB3')

print('\n=== FOREIGN YIELDS MONTHLY ===')
fetch_fred('BUND_M',    'IRLTLT01DEM156N')
fetch_fred('GILT_M',    'IRLTLT01GBM156N')
fetch_fred('BTP_M',     'IRLTLT01ITM156N')
fetch_fred('JGB_M',     'IRLTLT01JPM156N')

print('\n=== VOL ===')
fetch_fred('VIX',       'VIXCLS')
fetch_fred('EVZ',       'EVZCLS')

print('\n=== FUNDING STRESS ===')
fetch_fred('TED',       'TEDRATE')
fetch_fred('SOFR',      'SOFR')

print('\n=== OIL ===')
fetch_fred('WTI',       'DCOILWTICO')

=== TARGET PAIRS ===
EURUSD               DEXUSEU                  : 1999-01-04 -> 2026-05-08 (6,859 rows)
GBPUSD               DEXUSUK                  : 1990-01-02 -> 2026-05-08 (9,123 rows)
JPYUSD               DEXJPUS                  : 1990-01-02 -> 2026-05-08 (9,123 rows)

=== POLICY RATES ===
FED_FUNDS            FEDFUNDS                 : 1990-01-01 -> 2026-04-01 (436 rows)

=== US YIELDS ===
US_10Y               DGS10                    : 1990-01-02 -> 2026-05-13 (9,097 rows)
US_2Y                DGS2                     : 1990-01-02 -> 2026-05-13 (9,097 rows)
DTB3                 DTB3                     : 1990-01-02 -> 2026-05-13 (9,097 rows)

=== FOREIGN YIELDS MONTHLY ===
BUND_M               IRLTLT01DEM156N          : 1990-01-01 -> 2026-03-01 (435 rows)
GILT_M               IRLTLT01GBM156N          : 1990-01-01 -> 2026-03-01 (435 rows)
BTP_M                IRLTLT01ITM156N          : 1991-03-01 -> 2026-02-01 (420 rows)
JGB_M                IRLTLT01JPM156N          : 1990-0

In [1]:
from ecbdata import ecbdata as ecb
import requests
import time

print('=== ECB via ecbdata package ===')

def fetch_ecb_pkg(name, series_key):
    try:
        df = ecb.get_series(series_key, start='1999-01')
        df = df[df['OBS_VALUE'].notna()]
        if len(df) > 0:
            print(f"{name:20s} {series_key:45s}: "
                  f"{df['TIME_PERIOD'].iloc[0]} -> "
                  f"{df['TIME_PERIOD'].iloc[-1]} "
                  f"({len(df):,} rows)")
        else:
            print(f"{name:20s}: NO DATA")
    except Exception as e:
        print(f"{name:20s}: ERROR — {e}")

# ECB deposit facility rate
fetch_ecb_pkg('ECB_RATE',     'FM.B.U2.EUR.4F.KR.DFR.LEV')

# AAA euro area yield curve 10Y — proxy for Bund
fetch_ecb_pkg('BUND_AAA_10Y', 'YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y')

# All issuers euro area yield curve 10Y
fetch_ecb_pkg('EA_ALL_10Y',   'YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_10Y')

print()
print('=== BoE API ===')

def fetch_boe(name, series, delay=2):
    time.sleep(delay)
    try:
        url = (
            'https://www.bankofengland.co.uk/boeapps/database/'
            '_iadb-FromShowColumns.asp'
            f'?csv.x=yes&Datefrom=01/Jan/1975&Dateto=now'
            f'&SeriesCodes={series}&CSVF=TT&UsingCodes=Y'
        )
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code != 200:
            print(f"{name:20s}: HTTP {r.status_code}")
            return
        lines = r.text.strip().split('\n')
        data_lines = [l for l in lines
                      if l and l[0].isdigit() and ',' in l]
        if data_lines:
            first = data_lines[0].split(',')[0]
            last  = data_lines[-1].split(',')[0]
            print(f"{name:20s} {series:15s}: "
                  f"{first} -> {last} ({len(data_lines):,} rows)")
        else:
            print(f"{name:20s}: NO DATA — raw: {lines[:3]}")
    except Exception as e:
        print(f"{name:20s}: ERROR — {e}")

# BoE base rate — monthly
fetch_boe('BOE_RATE',  'IUMABEDR')

# Gilt 10Y spot rate — daily from 1993
fetch_boe('GILT_SPOT', 'IUDMNPY')

=== ECB via ecbdata package ===
ECB_RATE             FM.B.U2.EUR.4F.KR.DFR.LEV                    : 1999-01-01 -> 2025-06-11 (67 rows)
BUND_AAA_10Y         YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y          : 2004-09-06 -> 2026-05-14 (5,543 rows)
EA_ALL_10Y           YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_10Y          : 2004-09-06 -> 2026-05-14 (5,543 rows)

=== BoE API ===
BOE_RATE             IUMABEDR       : 31 Jan 1975 -> 30 Apr 2026 (616 rows)
GILT_SPOT            IUDMNPY        : 01 Nov 1993 -> 13 May 2026 (8,219 rows)
